# 05 - Hyperparameter Tuning

Tuning the 5 models from 04_Model_Training.ipynb. Reloading the cleaned dataset and redoing the same 70/30 split here (same random_state=42) since this is a separate notebook, no variables carried over. This keeps the split identical so results stay comparable to the baseline.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn.neural_network import MLPRegressor

In [2]:
df = pd.read_csv("../data/04_processed_dwsim/sobol_training_cleaned.csv")

input_columns = [
    "pressure_atm",
    "requested_vapor_fraction",
    "benzene_feed_fraction",
    "stages",
    "feed_stage_fraction",
    "reflux_ratio",
    "bottoms_fraction",
]

target_columns = [
    "x_D_benzene",
    "x_B_benzene",
    "Q_C",
    "Q_R",
]

X = df[input_columns]
y = df[target_columns]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.30, random_state=42
)

cv = KFold(n_splits=5, shuffle=True, random_state=42)
results = {}

## Random Forest Tuning

Default RF badly overfit purity earlier `(train R2 ~0.95, val R2 ~0.5-0.67)`, traced to `max_features=1.0` letting every split see all 7 features. Searching over max_features, max_depth, min_samples_leaf here to actually find the right regularization instead of picking one value by hand.

In [3]:
rf_param_dist = {
    "n_estimators": [200, 300, 500, 800],
    "max_depth": [8, 12, 16, 20, None],
    "min_samples_leaf": [1, 2, 3, 5],
    "max_features": ["sqrt", "log2", 0.5],
}

rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=rf_param_dist,
    n_iter=30,
    cv=cv,
    scoring="r2",
    random_state=42,
    n_jobs=-1,
)
rf_search.fit(X_train, y_train)
print("Best RF params:", rf_search.best_params_)

rf_best = rf_search.best_estimator_
pred_val = rf_best.predict(X_val)
results["RandomForest"] = {t: r2_score(y_val.iloc[:, i], pred_val[:, i]) for i, t in enumerate(target_columns)}
print(results["RandomForest"])

Best RF params: {'n_estimators': 500, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 16}
{'x_D_benzene': 0.8463702557000583, 'x_B_benzene': 0.7399770267918208, 'Q_C': 0.9712318661401669, 'Q_R': 0.9713184938154122}


## XGBoost Tuning

XGBoost already did well with hand-picked settings `(R2 0.99+ across all outputs)`. Checking here if that was close to optimal, or if learning_rate/depth/subsampling can push it further, and more importantly whether the tuned version generalizes better on the LHS holdout later, not just this val split.

In [4]:
xgb_param_dist = {
    "n_estimators": [200, 300, 500, 800],
    "max_depth": [3, 4, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
}

xgb_search = RandomizedSearchCV(
    xgb.XGBRegressor(random_state=42, n_jobs=-1),
    param_distributions=xgb_param_dist,
    n_iter=30,
    cv=cv,
    scoring="r2",
    random_state=42,
    n_jobs=-1,
)
xgb_search.fit(X_train, y_train)
print("Best XGB params:", xgb_search.best_params_)

xgb_best = xgb_search.best_estimator_
pred_val = xgb_best.predict(X_val)
results["XGBoost"] = {t: r2_score(y_val.iloc[:, i], pred_val[:, i]) for i, t in enumerate(target_columns)}
print(results["XGBoost"])

Best XGB params: {'subsample': 0.7, 'n_estimators': 500, 'max_depth': 4, 'learning_rate': 0.05, 'colsample_bytree': 1.0}
{'x_D_benzene': 0.9954139517765871, 'x_B_benzene': 0.9957946253790827, 'Q_C': 0.9989950196822022, 'Q_R': 0.9990398890095277}


## SVR Tuning

SVR needs scaled inputs, and since it doesn't do multi-output natively, it's wrapped in `MultiOutputRegressor`, so the search tunes one shared C/gamma/epsilon applied to each of the 4 output models. Searching the usual RBF knobs: C (regularization), gamma (kernel width), epsilon (margin).

In [5]:
scaler_X = StandardScaler().fit(X_train)
X_train_scaled = scaler_X.transform(X_train)
X_val_scaled = scaler_X.transform(X_val)

svr_param_dist = {
    "estimator__C": [1, 10, 50, 100, 200],
    "estimator__gamma": ["scale", "auto", 0.01, 0.1],
    "estimator__epsilon": [0.001, 0.01, 0.05, 0.1],
}

svr_search = RandomizedSearchCV(
    MultiOutputRegressor(SVR(kernel="rbf")),
    param_distributions=svr_param_dist,
    n_iter=20,
    cv=cv,
    scoring="r2",
    random_state=42,
    n_jobs=-1,
)
svr_search.fit(X_train_scaled, y_train)
print("Best SVR params:", svr_search.best_params_)

svr_best = svr_search.best_estimator_
pred_val = svr_best.predict(X_val_scaled)
results["SVR"] = {t: r2_score(y_val.iloc[:, i], pred_val[:, i]) for i, t in enumerate(target_columns)}
print(results["SVR"])

Best SVR params: {'estimator__gamma': 'scale', 'estimator__epsilon': 0.01, 'estimator__C': 200}
{'x_D_benzene': 0.9915231293336667, 'x_B_benzene': 0.9921065630323124, 'Q_C': 0.9947985681874393, 'Q_R': 0.9948660023765015}


## ANN Tuning

Both inputs and targets scaled for the MLP, same as notebook 04, targets inverse-transformed before scoring so R2 is in original units. Searching hidden layer size, `L2 (alpha)`, and initial `learning rate`. Kept n_iter small since MLP is the slowest to fit repeatedly under CV.

In [6]:
from sklearn.neural_network import MLPRegressor

scaler_y = StandardScaler().fit(y_train)
y_train_scaled = scaler_y.transform(y_train)
y_val_scaled = scaler_y.transform(y_val)

ann_param_dist = {
    "hidden_layer_sizes": [(64, 32), (128, 64), (100, 50, 25), (64, 64, 32)],
    "alpha": [0.0001, 0.001, 0.01],
    "learning_rate_init": [0.001, 0.003, 0.01],
}

ann_search = RandomizedSearchCV(
    MLPRegressor(max_iter=2000, early_stopping=True, random_state=42),
    param_distributions=ann_param_dist,
    n_iter=15,
    cv=cv,
    scoring="r2",
    random_state=42,
    n_jobs=-1,
)
ann_search.fit(X_train_scaled, y_train_scaled)
print("Best ANN params:", ann_search.best_params_)

ann_best = ann_search.best_estimator_
pred_val_scaled = ann_best.predict(X_val_scaled)
pred_val = scaler_y.inverse_transform(pred_val_scaled)
results["ANN"] = {t: r2_score(y_val.iloc[:, i], pred_val[:, i]) for i, t in enumerate(target_columns)}
print(results["ANN"])

Best ANN params: {'learning_rate_init': 0.01, 'hidden_layer_sizes': (128, 64), 'alpha': 0.001}
{'x_D_benzene': 0.9983628874076577, 'x_B_benzene': 0.9985169608948722, 'Q_C': 0.9992760729735904, 'Q_R': 0.9992608671520881}


## Tuned Model Comparison

Val R2 side by side for all 4 tuned models, same split throughout. Not re-tuning Polynomial Regression here, its only real knob is degree, which is a one-line check (2 vs 3) not a search, and degree 2 was already near ceiling in notebook 04. This table is a checkpoint, not final, still need the train->val->LHS gap and physical consistency checks before picking a winner per output.

In [7]:
results_df = pd.DataFrame(results).T
print("Tuned model validation R² comparison:\n")
print(results_df.round(4))

Tuned model validation R² comparison:

              x_D_benzene  x_B_benzene     Q_C     Q_R
RandomForest       0.8464       0.7400  0.9712  0.9713
XGBoost            0.9954       0.9958  0.9990  0.9990
SVR                0.9915       0.9921  0.9948  0.9949
ANN                0.9984       0.9985  0.9993  0.9993


## Polynomial/Linear Regression - Degree Check

OLS doesn't have the usual hyperparameters (no regularization, depth, kernel), so nothing to run through RandomizedSearchCV. The one real choice is degree, checking 2 vs 3 vs 4 manually to confirm degree 2 (used in notebook 04) is actually right and not leaving accuracy on the table or overfitting.

In [8]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

poly_results = {}

for degree in [2, 3, 4]:
    poly = PolynomialFeatures(degree=degree)
    X_train_poly = poly.fit_transform(X_train)
    X_val_poly = poly.transform(X_val)

    lin = LinearRegression()
    lin.fit(X_train_poly, y_train)

    pred_train = lin.predict(X_train_poly)
    pred_val = lin.predict(X_val_poly)

    print(f"\nDegree {degree}  (features: {X_train_poly.shape[1]})")
    row = {}
    for i, t in enumerate(target_columns):
        r2_train = r2_score(y_train.iloc[:, i], pred_train[:, i])
        r2_val = r2_score(y_val.iloc[:, i], pred_val[:, i])
        print(f"  {t:15s} train R² = {r2_train:.4f}   val R² = {r2_val:.4f}")
        row[t] = r2_val
    poly_results[f"degree_{degree}"] = row

print("\nSummary:")
print(pd.DataFrame(poly_results).T.round(4))


Degree 2  (features: 36)
  x_D_benzene     train R² = 0.9806   val R² = 0.9798
  x_B_benzene     train R² = 0.9802   val R² = 0.9789
  Q_C             train R² = 0.9997   val R² = 0.9997
  Q_R             train R² = 0.9998   val R² = 0.9998

Degree 3  (features: 120)
  x_D_benzene     train R² = 0.9751   val R² = 0.9739
  x_B_benzene     train R² = 0.9779   val R² = 0.9759
  Q_C             train R² = 0.9994   val R² = 0.9994
  Q_R             train R² = 0.9995   val R² = 0.9995

Degree 4  (features: 330)
  x_D_benzene     train R² = 0.9413   val R² = 0.9362
  x_B_benzene     train R² = 0.9601   val R² = 0.9574
  Q_C             train R² = 0.9968   val R² = 0.9963
  Q_R             train R² = 0.9970   val R² = 0.9964

Summary:
          x_D_benzene  x_B_benzene     Q_C     Q_R
degree_2       0.9798       0.9789  0.9997  0.9998
degree_3       0.9739       0.9759  0.9994  0.9995
degree_4       0.9362       0.9574  0.9963  0.9964


### Degree selection

Degree 2 beats degree 3 and 4 on every output, R2 drops as degree goes up `(x_D_benzene: 0.9798 -> 0.9739 -> 0.9362)`. This is overfitting from feature explosion, degree 2 gives 35 features from 7 inputs, degree 3 ~120, degree 4 ~330. With only 1766 training rows, the higher-degree models start fitting noise instead of real structure.

**Degree 2 locked in as final.**

In [9]:
best_degree = 2

poly = PolynomialFeatures(degree=best_degree)
X_train_poly = poly.fit_transform(X_train)
X_val_poly = poly.transform(X_val)

lin_final = LinearRegression()
lin_final.fit(X_train_poly, y_train)
pred_val = lin_final.predict(X_val_poly)

results["PolynomialRegression"] = {
    t: r2_score(y_val.iloc[:, i], pred_val[:, i]) for i, t in enumerate(target_columns)
}

results_df = pd.DataFrame(results).T
print("Full tuned model comparison (all 5):\n")
print(results_df.round(4))

Full tuned model comparison (all 5):

                      x_D_benzene  x_B_benzene     Q_C     Q_R
RandomForest               0.8464       0.7400  0.9712  0.9713
XGBoost                    0.9954       0.9958  0.9990  0.9990
SVR                        0.9915       0.9921  0.9948  0.9949
ANN                        0.9984       0.9985  0.9993  0.9993
PolynomialRegression       0.9798       0.9789  0.9997  0.9998


## Final Tuned Comparison - All 5 Models

| Model | x_D_benzene | x_B_benzene | Q_C | Q_R |
|---|---|---|---|---|
| Random Forest | 0.8464 | 0.7400 | 0.9712 | 0.9713 |
| XGBoost | 0.9954 | 0.9958 | 0.9990 | 0.9990 |
| SVR | 0.9915 | 0.9921 | 0.9948 | 0.9949 |
| ANN | 0.9984 | 0.9985 | 0.9993 | 0.9993 |
| Polynomial Regression (deg 2) | 0.9798 | 0.9789 | 0.9997 | 0.9998 |

A few things stand out:

- `Random Forest` is clearly the weakest here even after tuning, tree ensembles don't handle the smooth, saturating purity curves as well as boosting or a continuous approximator like ANN/SVR. Unlikely to be the final pick for x_D/x_B.
- `Polynomial Regression` is surprisingly strong on Q_C/Q_R (0.9997 / 0.9998, best of all 5 on duty), matches the earlier correlation analysis showing duty is close to linear/mildly quadratic in reflux ratio. Noticeably behind on purity though.
- `ANN, XGBoost, SVR` are all within ~0.005-0.01 of each other on every output, basically tied on accuracy. So accuracy alone can't decide it, robustness on the LHS holdout and physical consistency of predictions are what should actually separate them.

**Saving the tuned validation results to CSV so they can be reused in the final holdout evaluation notebook**

In [10]:
results_df = pd.DataFrame(results).T
results_df.to_csv("../data/05_tuning_results/validation_r2_scores.csv", index_label="model")

## Saving the best parameters for each model as a csv

In [11]:
import json

best_params = {
    "RandomForest": rf_search.best_params_,
    "XGBoost": xgb_search.best_params_,
    "SVR": svr_search.best_params_,
    "ANN": ann_search.best_params_,
    "PolynomialRegression": {"degree": best_degree},
}

with open("../data/05_tuning_results/best_params.json", "w") as f:
    json.dump(best_params, f, indent=2)

## Saving the trained models as .pkl files, for use in the final holdout

In [12]:

import joblib

model_dir = "../models/01_trained_models_all_5"

joblib.dump(rf_best, f"{model_dir}/random_forest.pkl")
joblib.dump(xgb_best, f"{model_dir}/xgboost.pkl")
joblib.dump(svr_best, f"{model_dir}/svr.pkl")
joblib.dump(ann_best, f"{model_dir}/ann.pkl")
joblib.dump(lin_final, f"{model_dir}/polynomial_regression.pkl")

joblib.dump(scaler_X, f"{model_dir}/scaler_X.pkl")
joblib.dump(scaler_y, f"{model_dir}/scaler_y.pkl")
joblib.dump(poly, f"{model_dir}/poly_features.pkl")


['../models/01_trained_models_all_5/poly_features.pkl']